In [ ]:
# Notebooks live in notebooks/; make the repo root importable (ppo, train_ppo, ...).
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "ppo" / "paths.py").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


# `plot_trajectories` — usage walkthrough

Overlay GA + PPO FBS trajectories on common axes.

Inputs the plotter accepts (auto-detected):

| spec | resolves to |
|------|-------------|
| `ga_runs/run_NNN` | `trajectory.csv` if present, else `result.mat` |
| `ga_runs/run_NNN/result.mat` | reads `history.bestIndividuals` + `mbs_params` |
| `ga_runs/run_NNN/trajectory.csv` | flat per-generation best individuals |
| `test_logs/<X-Y-Z>/<stem>` | auto-appends `_trajectory.csv` |
| `test_logs/<X-Y-Z>/<stem>_trajectory.csv` | PPO rollout |

**Constraint:** the plotter refuses to mix 1-FBS and 2-FBS trajectories in one figure (raises `ValueError`).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import scipy

from plot_trajectories import (
    plot_trajectories,
    load_trajectory,
    REPO_ROOT,
    GA_RUNS,
    RL_TEST_LOGS,
)

## 1. Discover what's available

In [ ]:
ga_runs = sorted(p.name for p in GA_RUNS.glob("run_*") if p.is_dir())
print("GA runs:")
for r in ga_runs[-10:]:
    print(" ", r)

print("\nPPO test trajectories (by code):")
for code_dir in sorted(p for p in RL_TEST_LOGS.glob("*") if p.is_dir()):
    trajs = sorted(code_dir.glob("*_trajectory.csv"))
    print(f"  [{code_dir.name}]")
    for t in trajs[-3:]:
        print("    ", t.name.replace("_trajectory.csv", ""))

## 2. Inspect a single trajectory before plotting

`load_trajectory(...)` returns a `Trajectory` dataclass — useful for sanity-checking shape, num_fbs, and detected MBS coordinates.

In [ ]:
ga_traj = load_trajectory("ga_runs/run_001")
print("name      :", ga_traj.name)
print("source    :", ga_traj.source)
print("num_fbs   :", ga_traj.num_fbs)
print("code      :", ga_traj.code)
print("mbs (x,y) :", ga_traj.mbs_x, ga_traj.mbs_y)
print("shape     :", ga_traj.df.shape)
ga_traj.df.head()

## 3. Plot one trajectory

The simplest case — pass a single spec. Path is colored by step/generation; square = start, star = end. MBS positions are auto-detected when known.

In [ ]:
plot_trajectories(
    [("ga_runs/run_003", "GA run_003")],
    title="GA only",
)
plt.show()

## 4. Overlay GA + PPO

Edit the cell below so the two specs match the same X-Y-Z setup (e.g. both 1-FBS / 1-MBS = 1-1-*). The plotter will refuse if num_fbs differs.

Source styles: GA solid, PPO dashed. Each trajectory gets its own color.

In [ ]:
# specs = [
#     ("ga_runs/run_002", "GA A"), #1-1-1
#     ("ga_runs/run_003", "GA B"), #1-1-2
#     ("test_logs/1-1-1/run_044_20260501_124857", "PPO A"), #1-1-1
#     ("test_logs/1-1-2/run_039_20260501_124158", "PPO B"), #1-1-2
# ]
specs = [
    ("ga_runs/run_001", "GA"), #2-1-1
    ("test_logs/2-1-1/run_045_20260501_125458", "PPO"), #2-1-1
]

# GA_PICK = "ga_runs/run_002"
# RL_PICK = "test_logs/1-1-1/run_044_20260501_124857"

# check arguments to plot_trajectories:
fig, ax = plot_trajectories(
    specs, 
    title="GA vs PPO  (2-1-1)",
    color_by="power",
    mark_power_status=True)



plt.show()

## 5. Compare several runs of the same kind

Useful for visualizing variance: e.g. multiple GA runs of the same X-Y-Z setup.

In [ ]:
# Pick 2-3 GA runs that share num_fbs (otherwise the plotter raises).
specs = [
    ("ga_runs/run_002", "GA #1"),
    ("ga_runs/run_003", "GA #2"),
    # ("ga_runs/run_003", "GA #3"),
]
plot_trajectories(specs, title="GA -- repeat runs")
plt.show()

## 6. Tweak rendering

Common knobs:

- `color_by_step=False` — flat color per trajectory; cleaner for many overlays.
- `show_mbs=False` — hide MBS markers.
- `annotate_endpoints=False` — drop start/end glyphs.
- `figsize` / `xlim` / `ylim` — sizing & cropping.

In [ ]:
fig, ax = plot_trajectories(
    [
        ("ga_runs/run_001", "GA"),
        ("test_logs/1-1-1/run_044_20260501_124857", "PPO"),
    ],
    title="Flat-color version",
    color_by_step=False,
    figsize=(7, 5.5),
    annotate_endpoints=True,
)
plt.show()

## 7. Plot into an existing figure / save to disk

Pass `ax=` to embed in a multi-panel figure. Use `fig.savefig(...)` to write a PNG / PDF.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

plot_trajectories(
    [("ga_runs/run_001", "GA")],
    title="GA only",
    ax=axes[0],
)
plot_trajectories(
    [("test_logs/1-1-1/run_044_20260501_124857", "PPO")],
    title="PPO only",
    ax=axes[1],
)

fig.tight_layout()
out = REPO_ROOT / "trajectory_panels.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print("saved ->", out)
plt.show()

## 8. Sanity check — the mixed-FBS guard

If the specs disagree on `num_fbs`, the plotter raises a `ValueError`. Useful to verify.

In [ ]:
# Replace these with two runs that genuinely have different num_fbs to see the guard fire.
# Otherwise just leave commented out.

# try:
#     plot_trajectories([
#         ("ga_runs/run_001", "GA 1FBS"),     # num_fbs = 1
#         ("ga_runs/run_007", "GA 2FBS"),     # num_fbs = 2
#     ])
# except ValueError as e:
#     print("Guard fired:", e)

## 9. CLI equivalent

Same plot, no notebook:

```bash
python plot_trajectories.py \
    ga_runs/run_001 \
    test_logs/1-1-1/run_044_20260501_124857_trajectory.csv \
    --label GA --label PPO \
    --title "GA vs PPO  (1-1-1)" \
    --save trajectory_compare.png
```